In [1]:
import pandas as pd
import numpy as np
import sys
import os

In [8]:
breakpoints = pd.read_csv("../example/interactions.tsv",sep="\t",names=["seqid1","seqid2","pos1","pos2","count","junction1","junction2","gene1","gene2"])
breakpoints.head()

,seqid1,seqid2,pos1,pos2,count,junction1,junction2,gene1,gene2
0,7,M33262.1,18888431,1240,23,vif,ENSMMUG00000001534,ENSMMUG00000001534,env;env/vpu;gag;nef;rev;tat;vif;vpr
1,7,M33262.1,18888431,1240,29,vif,ENSMMUG00000001534,ENSMMUG00000001534,env;env/vpu;gag;nef;rev;tat;vif;vpr
2,20,M33262.1,43612782,10533,38,-,-,-,env;env/vpu;gag;nef;rev;tat;vif;vpr
3,20,M33262.1,43612782,1072,29,-,-,-,env;env/vpu;gag;nef;rev;tat;vif;vpr
4,20,M33262.1,43612782,10533,8,-,-,-,env;gag;nef;rev;tat;vif;vpr


In [36]:
def custom_grouping(row):
    if row['gene1'] != '-':
        return (row['gene1'], row['seqid1'])
    else:
        return ((row['pos1'] - 1) // 100000, row['seqid1'])


genes = breakpoints[["seqid1","pos1","gene1"]]

# Apply the custom grouping function
genes['group'] = genes.apply(custom_grouping, axis=1)

# Group by the 'group' column
genes = genes.groupby('group').agg({'pos1': 'min', 'gene1': 'first', 'seqid1': 'first'}).reset_index(drop=True)
genes[["seqid1","pos1","gene1"]].to_csv("../example/m.mulatta.genes.bed",sep="\t",index=False,header=False)

/var/folders/kg/dntg2ryj6bb7y29gqk4q0r_c0000gn/T/ipykernel_2162/3233406034.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  genes['group'] = genes.apply(custom_grouping, axis=1)


In [37]:
def custom_grouping(row):
    return ((row['pos1'] - 1) // 100000, row['seqid1'], (row['pos2'] - 1) // 10, row['seqid2'])

# Extract relevant columns
interactions = breakpoints[["seqid1", "seqid2", "pos1", "pos2", "count"]]

# Apply the custom grouping function
interactions['group'] = interactions.apply(custom_grouping, axis=1)

# Group by the 'group' column and sum the 'count' column
interactions = interactions.groupby('group').agg({
    'pos1': 'min',
    'pos2': 'min',
    'seqid1': 'first',
    'seqid2': 'first',
    'count': 'sum'
}).reset_index()

interactions[["seqid1","seqid2","pos1","pos2","count"]].to_csv("../example/m.mulatta.interactions.tsv",sep="\t",index=False,header=False)

/var/folders/kg/dntg2ryj6bb7y29gqk4q0r_c0000gn/T/ipykernel_2162/88680528.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  interactions['group'] = interactions.apply(custom_grouping, axis=1)
